In [0]:
from pyspark.sql.functions import sum, avg, round

# monthly sales trend
monthly_sales = spark.read.table("ecommerce.e_comm_gold.factSales")
monthly_sales.createOrReplaceTempView("monthly_sales")

# calculate metrics 
agg_monthly_sales = spark.sql("""
        
        select 
            year(order_date) as order_year_id
            , month(order_date) as order_month_id
            , count(order_id) as total_orders
            , sum(order_quantity) as total_items_sold
            , round(sum(sale_amount),2) as total_revenue
            , round(avg(sale_amount),2) as avg_order_value
            , current_timestamp() as load_ts
        from monthly_sales
        where dq_note = "is_valid"
        group by order_year_id, order_month_id
        """)
    
# write to delta table

agg_monthly_sales.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.fact_agg_monthly_sales")